In [1]:
import math
from typing import Callable
from statistics import NormalDist

import numpy as np

def prom_update(prom_prev: float, x_new: float, n_curr: int):
    if n_curr < 1:
        raise Exception()
    elif n_curr == 1:
        return x_new
    else:
        return prom_prev + (x_new - prom_prev) / n_curr


def s_cuad_update(s_cuad_prev: float, prom_prev: float, prom_curr: float, n_curr: int):
    if n_curr < 1:
        raise Exception()
    elif n_curr == 1:
        return 0.0
    else:
        return (1 - (1/(n_curr - 1))) * s_cuad_prev + n_curr * ((prom_curr - prom_prev)**2)


def calculo_z_standard(alpha: float):
    return NormalDist().inv_cdf(1 - alpha / 2)

def intervalo(prom: float, s_cuad: float, n: int, alpha: float):
    z_alpha_2 = calculo_z_standard(alpha)

    std = math.sqrt(s_cuad/n)
    izq = prom - z_alpha_2 * std
    der = prom + z_alpha_2 * std

    return izq, der


### Ejercicio 11.
Considerar el modelo de reparación.

a. Desarrollar un programa que simule el proceso, registrando los tiempos de falla, el número de máquinas de repuesto y en reparación.

b. Utilizar el programa para estimar el tiempo medio hasta la falla del sistema en el caso en que el número $n$ de máquinas en funcionamiento sea $n = 6$, el stock inicial de máquinas de repuesto sea $s = 4$ y las distribuciones de los tiempos de funcionamiento y reparación están dadas por
$$ F(t) = 1 - e^{-2t}, \quad G(t) = 1 - e^{-3t}, \quad t \ge 0 $$
Detener las simulaciones cuando la desviación estándar muestral del estimador de la media sea menor que $0.01$.

c. Construir un intervalo de confianza del $95 \%$ para el tiempo medio hasta la falla del sistema.

d. Estimar la probabilidad de que el sistema falle antes de los $90$ minutos. Detener las simulaciones cuando la desviación estándar muestral del estimador de la proporción sea menor que $0.01$.

e. Construir un intervalo de confianza del $95 \%$ para la probabilidad estimada en el inciso anterior.


---

In [2]:
# Punto A
def punto_a(n: int, s: int, t_funcionamiento: Callable[[], float], t_arreglo: Callable[[], float]):
    # Registros
    F = [] # Tiempo de falla
    R = [] # Maquinas en repuesto y reparación
    R_T = [] # Tiempo en la cual se registro las maquinas en repuesto y reparación

    # Inicialización
    t, r = 0, 0

    T = []
    for _ in range(n):
        T.append(t + t_funcionamiento())
    T.sort()

    t_estrella = math.inf
    
    while r <= s:
        evento_prox = min(min(T), t_estrella)
        
        if evento_prox in T:  # Fallo una maquina
            t = evento_prox
            r += 1

            i = T.index(evento_prox)
            
            if r > s:
                break
            
            if r == 1:
                t_estrella = t + t_arreglo()
    
            T[i] = math.inf
            
            F.append(t)
    
        else: # Se reparo una maquina
            t = t_estrella
            j = T.index(math.inf)
            r -= 1
            
            T[j] = t + t_funcionamiento()
            
            if r > 0:
                t_estrella = t + t_arreglo()
            else:
                t_estrella = math.inf
            
            T.sort()

        R.append((n - r, r))
        R_T.append(t)

    return t, F, R, R_T

In [3]:
# Punto B

def t_funcionamiento():
    return -math.log(1 - np.random.random()) / 2


def t_arreglo():
    return -math.log(1 - np.random.random()) / 3

def generar_X_i_b():
    t, _, _, _ = punto_a(6, 4, t_funcionamiento, t_arreglo)
    
    return t

def punto_b():
    media = generar_X_i_b()
    s_cuad, n = 0, 1

    while n < 100 or math.sqrt(s_cuad / n) >= 0.01:
        X = generar_X_i_b()
        n += 1

        media_prev = media
        media = prom_update(media, X, n)
        s_cuad = s_cuad_update(s_cuad, media_prev, media, n)

    print(f"Cantidad de datos generados: ", n)
    print(f"Tiempo medio hasta la falla del sistema: {media:.4f}")
    print(f"Desviación estándar muestral = {math.sqrt(s_cuad / n):.4f} ")


punto_b()

Cantidad de datos generados:  5074
Tiempo medio hasta la falla del sistema: 1.0791
Desviación estándar muestral = 0.0100 


In [4]:
# Punto C

def sim_media(l: float, alpha: float):
    # l = semi-ancho = L / 2

    z_alpha_2 = calculo_z_standard(alpha)
    cota = l / (z_alpha_2 * 2)

    promedio = generar_X_i_b()
    s_cuad, n = 0, 1

    while n <= 100 or math.sqrt(s_cuad / n) >= cota:
        X = generar_X_i_b()
        n += 1

        promedio_prev = promedio
        promedio = prom_update(promedio, X, n)
        s_cuad = s_cuad_update(s_cuad, promedio_prev, promedio, n)

    i_d, i_i = intervalo(promedio, s_cuad, n, alpha)

    print(f"{"Amplitud":>9} | {"N° Sim":>9} | {"Intervalo obtenido":>18} | {"STDM":>10} | {"Aprox":>10}")
    print(f"{l:>9} | {n:9d} | {f"[{i_d:.4f}, {i_i:.4f}]":>18} | {math.sqrt(s_cuad / n):10.4f} | {promedio:10.4f}")


l = 0.1
alpha = 0.05
sim_media(l, alpha)

 Amplitud |    N° Sim | Intervalo obtenido |       STDM |      Aprox
      0.1 |       726 |   [1.0295, 1.1294] |     0.0255 |     1.0794


In [5]:
# Punto D

def generar_X_i_d():
    t, _, _, _ = punto_a(6, 4, t_funcionamiento, t_arreglo)

    return int(t < 0.9)

def punto_d():
    p = generar_X_i_d()
    n = 1

    while n < 100 or math.sqrt(p * (1 - p) / n) >= 0.01:
        X = generar_X_i_d()
        n += 1

        p = prom_update(p, X, n)

    print(f"Cantidad de datos generados: ", n)
    print(f"Prob de que el sistema falle antes de los 90 minutos: {p:.4f}")
    print(f"Desviación estándar muestral = {math.sqrt(p * (1 - p) / n):.4f} ")


punto_d()

Cantidad de datos generados:  2499
Prob de que el sistema falle antes de los 90 minutos: 0.5130
Desviación estándar muestral = 0.0100 


In [6]:
# Punto E

def sim_prob(l: float, alpha: float):
    z_alpha_2 = calculo_z_standard(alpha)
    cota = l / (z_alpha_2 * 2)

    p = generar_X_i_d()
    n = 1

    while n <= 100 or math.sqrt(p*(1 - p) / n) >= cota:
        X = generar_X_i_d()
        n += 1

        p = prom_update(p, X, n)

    i_d, i_i = intervalo(p, math.sqrt(p*(1 - p) / n), n, alpha)

    print(f"{"Amplitud":>9} | {"N° Sim":>9} | {"Intervalo obtenido":>18} | {"STDM":>10} | {"Aprox":>10}")
    print(f"{l:>9} | {n:9d} | {f"[{i_d:.4f}, {i_i:.4f}]":>18} | {math.sqrt(p*(1 - p) / n):10.4f} | {p:10.4f}")

l = 0.1
alpha = 0.05
sim_prob(l, alpha)

 Amplitud |    N° Sim | Intervalo obtenido |       STDM |      Aprox
      0.1 |       384 |   [0.4658, 0.4977] |     0.0255 |     0.4818
